# R-(+)-limonene in *Penicillium digitatum*: a publisher-table audit

This notebook audits the checksum-pinned publisher workbook for Zhang et al. (2020), Table S7. It reads author-provided expression values and annotations; it does not estimate differential expression or identify a limonene-binding target. The comparison is **P_L_12h versus P_12h**, both at 12 hours.

In [1]:
from pathlib import Path
import csv, hashlib, json, math, re
from openpyxl import load_workbook

source = Path("research/penicillium-limonene-supplement/data/Data_Sheet_1.xlsx")
expected = "85fac5c96d733d35402a67f1f3af7ac364f4528c8ed37cd1292bc21da0639700"
actual = hashlib.sha256(source.read_bytes()).hexdigest()
assert actual == expected, f"Publisher workbook hash changed: {actual}"
book = load_workbook(source, read_only=True, data_only=True)
print("SHA-256:", actual)
print("Sheets:", ", ".join(book.sheetnames))

SHA-256: 85fac5c96d733d35402a67f1f3af7ac364f4528c8ed37cd1292bc21da0639700
Sheets: Table S1, Table S2, Table S3, Table S5 daixiu, Table S4, Table S5, Table S6, Table S7, Fig.S1, Fig. S2


## Extraction and checks

Rows 1–2 of Table S7 contain the title and column labels. Two trailing rows are blank. Each retained row must have a unique gene ID and numeric, nonnegative expression values. The workbook does not supply replicate values or per-gene adjusted p values in this table.

In [2]:
sheet = book["Table S7"]
headers = list(next(sheet.iter_rows(min_row=2, max_row=2, values_only=True)))
expected_headers = ["Gene ID", "NR Annotation", "KEGG", "P_0h-Expression", "P_12h-Expression", "P_L_12h-Expression"]
assert headers == expected_headers, headers
rows = []
for values in sheet.iter_rows(min_row=3, values_only=True):
    if not values[0]:
        continue
    gene, nr, kegg, zero, control, exposed = values[:6]
    expression = [float(value) for value in (zero, control, exposed)]
    assert all(math.isfinite(value) and value >= 0 for value in expression)
    rows.append({"gene_id": str(gene), "nr_annotation": str(nr or ""), "kegg_annotation": str(kegg or ""),
                 "fpkm_0h": expression[0], "fpkm_12h_control": expression[1], "fpkm_12h_limonene": expression[2]})
assert len({row["gene_id"] for row in rows}) == len(rows)
print("Retained gene rows:", len(rows))

Retained gene rows: 58


## Descriptive response and annotation audit

Direction compares the two 12-hour expression columns. A 0.1 FPKM pseudocount makes the log ratio defined, including low-expression rows. The P450 label check is a literal string audit of the supplied NR and KEGG fields, not a functional annotation, protein-domain analysis, or target validation.

In [3]:
for row in rows:
    row["log2_fpkm_ratio_12h"] = round(math.log2((row["fpkm_12h_limonene"] + 0.1) / (row["fpkm_12h_control"] + 0.1)), 6)
    row["nr_p450_label"] = bool(re.search(r"p450|\bcyp", row["nr_annotation"], re.I))
    row["kegg_p450_label"] = bool(re.search(r"cytochrome p450", row["kegg_annotation"], re.I))
    row["any_explicit_p450_label"] = row["nr_p450_label"] or row["kegg_p450_label"]

summary = {
    "source_sha256": actual, "sheet": "Table S7", "unique_gene_rows": len(rows),
    "higher_at_12h_exposed": sum(row["fpkm_12h_limonene"] > row["fpkm_12h_control"] for row in rows),
    "lower_at_12h_exposed": sum(row["fpkm_12h_limonene"] < row["fpkm_12h_control"] for row in rows),
    "equal_at_12h": sum(row["fpkm_12h_limonene"] == row["fpkm_12h_control"] for row in rows),
    "nr_explicit_p450_label": sum(row["nr_p450_label"] for row in rows),
    "kegg_explicit_p450_label": sum(row["kegg_p450_label"] for row in rows),
    "either_explicit_p450_label": sum(row["any_explicit_p450_label"] for row in rows),
    "per_gene_adjusted_p_values_available_in_s7": False,
    "effect_class": "author-provided FPKM, descriptive ratio only",
}
out = Path("research/penicillium-limonene-supplement/results")
out.mkdir(parents=True, exist_ok=True)
(out / "analysis_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
with (out / "table_s7_audit.tsv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]), delimiter="\t")
    writer.writeheader()
    writer.writerows(rows)
print(json.dumps(summary, indent=2))

{
  "source_sha256": "85fac5c96d733d35402a67f1f3af7ac364f4528c8ed37cd1292bc21da0639700",
  "sheet": "Table S7",
  "unique_gene_rows": 58,
  "higher_at_12h_exposed": 31,
  "lower_at_12h_exposed": 27,
  "equal_at_12h": 0,
  "nr_explicit_p450_label": 8,
  "kegg_explicit_p450_label": 6,
  "either_explicit_p450_label": 10,
  "per_gene_adjusted_p_values_available_in_s7": false,
  "effect_class": "author-provided FPKM, descriptive ratio only"
}


## Interpretation

The direction count recovers the publisher’s 31 higher and 27 lower candidate rows for the time-matched comparison. Because Table S7 has neither replicate-level measurements nor per-gene adjusted p values, the notebook cannot independently confirm the paper’s DEG threshold or statistical uncertainty. Most candidate rows lack an explicit P450/CYP label in the supplied annotation fields; absence of that text does not exclude P450 activity. The transcriptional response cannot establish which enzyme converts limonene to α-terpineol. This study and the separate linalool studies differ in compound and organism, so their pathway responses do not constitute cross-study replication.